In [17]:
import sys
from pyspark.sql import SparkSession, functions as sf
from pyspark.sql.window import Window
import shutil
from google.colab import files

In [18]:
POSTS_XML = "/content/posts_sample.xml"
LANGS_CSV = "/content/programming-languages.csv"
PARQUET_OUT = "/content/top_languages_2010_2020.parquet"

In [19]:
spark = (
    SparkSession.builder
    .appName("StackOverflowLanguageReport")
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.0.2


In [20]:
langs_ref = (
    spark.read
    .option("header", True)
    .csv(LANGS_CSV)
)

name_field = langs_ref.columns[0]

langs_ref = (
    langs_ref
    .select(
        sf.trim(sf.lower(sf.col(name_field))).alias("language_name")
    )
    .dropna()
    .dropDuplicates()
)

print("Количество языков:", langs_ref.count())
langs_ref.show(20, truncate=False)

Количество языков: 698
+-----------------------------------+
|language_name                      |
+-----------------------------------+
|ceylon                             |
|hope                               |
|m#                                 |
|metafont                           |
|mortran                            |
|pl-11                              |
|q (equational programming language)|
|bash and                           |
|objectlogo                         |
|reia                               |
|sail                               |
|comal                              |
|euler                              |
|karel                              |
|lithe                              |
|nwscript                           |
|xmos architecture )                |
|axum                               |
|common lisp (also known as cl)     |
|falcon                             |
+-----------------------------------+
only showing top 20 rows


In [21]:
xml_lines = spark.read.text(POSTS_XML)

posts_with_tags = (
    xml_lines
    .filter(
        sf.col("value").contains("CreationDate=") &
        sf.col("value").contains("Tags=")
    )
)

print("Подходящих XML-строк:", posts_with_tags.count())

Подходящих XML-строк: 18057


In [22]:
parsed_posts = (
    posts_with_tags
    .select(
        sf.regexp_extract("value", r'CreationDate="(\d{4})', 1).cast("int").alias("year_num"),
        sf.regexp_extract("value", r'Tags="([^"]*)"', 1).alias("tags_blob")
    )
    .filter(sf.col("year_num").between(2010, 2020))
)

print("Записей за 2010-2020:", parsed_posts.count())
parsed_posts.show(5, truncate=False)

Записей за 2010-2020: 17644
+--------+--------------------------------------------------------------+
|year_num|tags_blob                                                     |
+--------+--------------------------------------------------------------+
|2010    |&lt;c++&gt;&lt;character-encoding&gt;                         |
|2010    |&lt;sharepoint&gt;&lt;infopath&gt;                            |
|2010    |&lt;iphone&gt;&lt;app-store&gt;&lt;in-app-purchase&gt;        |
|2010    |&lt;symfony1&gt;&lt;schema&gt;&lt;doctrine&gt;&lt;fixtures&gt;|
|2010    |&lt;java&gt;                                                  |
+--------+--------------------------------------------------------------+
only showing top 5 rows


In [23]:
tag_rows = (
    parsed_posts
    .withColumn("tags_blob", sf.regexp_replace("tags_blob", r"&lt;|&gt;", " "))
    .withColumn("tags_blob", sf.regexp_replace("tags_blob", r"\s+", " "))
    .withColumn("tag_item", sf.explode(sf.split(sf.trim(sf.col("tags_blob")), " ")))
    .withColumn("tag_item", sf.trim(sf.lower(sf.col("tag_item"))))
    .filter(sf.col("tag_item") != "")
    .select("year_num", "tag_item")
)

print("Тегов после explode:", tag_rows.count())
tag_rows.show(10, truncate=False)

Тегов после explode: 52118
+--------+------------------+
|year_num|tag_item          |
+--------+------------------+
|2010    |c++               |
|2010    |character-encoding|
|2010    |sharepoint        |
|2010    |infopath          |
|2010    |iphone            |
|2010    |app-store         |
|2010    |in-app-purchase   |
|2010    |symfony1          |
|2010    |schema            |
|2010    |doctrine          |
+--------+------------------+
only showing top 10 rows


In [24]:
language_mentions = (
    tag_rows.alias("t")
    .join(
        langs_ref.alias("l"),
        sf.col("t.tag_item") == sf.col("l.language_name"),
        how="inner"
    )
    .select(
        sf.col("t.year_num").alias("report_year"),
        sf.col("l.language_name").alias("language")
    )
)

print("Тегов, совпавших со справочником:", language_mentions.count())
language_mentions.show(10, truncate=False)

Тегов, совпавших со справочником: 8054
+-----------+-----------+
|report_year|language   |
+-----------+-----------+
|2010       |java       |
|2010       |php        |
|2010       |ruby       |
|2010       |c          |
|2010       |php        |
|2010       |python     |
|2010       |javascript |
|2010       |applescript|
|2010       |php        |
|2010       |php        |
+-----------+-----------+
only showing top 10 rows


In [25]:
year_language_stats = (
    language_mentions
    .groupBy("report_year", "language")
    .agg(sf.count("*").alias("mentions"))
)

year_language_stats.createOrReplaceTempView("year_language_stats")
year_language_stats.orderBy("report_year", sf.desc("mentions")).show(20, truncate=False)

+-----------+------------+--------+
|report_year|language    |mentions|
+-----------+------------+--------+
|2010       |java        |52      |
|2010       |php         |46      |
|2010       |javascript  |44      |
|2010       |python      |26      |
|2010       |objective-c |23      |
|2010       |c           |20      |
|2010       |ruby        |12      |
|2010       |delphi      |8       |
|2010       |r           |3       |
|2010       |applescript |3       |
|2010       |perl        |3       |
|2010       |bash        |3       |
|2010       |haskell     |2       |
|2010       |sed         |2       |
|2010       |f#          |2       |
|2010       |xpath       |1       |
|2010       |racket      |1       |
|2010       |actionscript|1       |
|2010       |mouse       |1       |
|2010       |scheme      |1       |
+-----------+------------+--------+
only showing top 20 rows


In [26]:
top10_sql = """
SELECT
    report_year,
    language,
    mentions,
    ROW_NUMBER() OVER (
        PARTITION BY report_year
        ORDER BY mentions DESC, language ASC
    ) AS place
FROM year_language_stats
"""

ranked_languages = (
    spark.sql(top10_sql)
    .filter(sf.col("place") <= 10)
)

ranked_languages.orderBy("report_year", "place").show(50, truncate=False)

+-----------+-----------+--------+-----+
|report_year|language   |mentions|place|
+-----------+-----------+--------+-----+
|2010       |java       |52      |1    |
|2010       |php        |46      |2    |
|2010       |javascript |44      |3    |
|2010       |python     |26      |4    |
|2010       |objective-c|23      |5    |
|2010       |c          |20      |6    |
|2010       |ruby       |12      |7    |
|2010       |delphi     |8       |8    |
|2010       |applescript|3       |9    |
|2010       |bash       |3       |10   |
|2011       |php        |102     |1    |
|2011       |java       |93      |2    |
|2011       |javascript |83      |3    |
|2011       |python     |37      |4    |
|2011       |objective-c|34      |5    |
|2011       |c          |24      |6    |
|2011       |ruby       |20      |7    |
|2011       |perl       |9       |8    |
|2011       |delphi     |8       |9    |
|2011       |bash       |7       |10   |
|2012       |php        |154     |1    |
|2012       |jav

In [27]:
report_compact = (
    ranked_languages
    .groupBy("report_year")
    .pivot("place", list(range(1, 11)))
    .agg(sf.first("language"))
    .orderBy("report_year")
)

print("Pivot таблица:")

report_wide = report_compact.select(
    sf.col("report_year").alias("year"),
    sf.col("1").alias("1"),
    sf.col("2").alias("2"),
    sf.col("3").alias("3"),
    sf.col("4").alias("4"),
    sf.col("5").alias("5"),
    sf.col("6").alias("6"),
    sf.col("7").alias("7"),
    sf.col("8").alias("8"),
    sf.col("9").alias("9"),
    sf.col("10").alias("10"),
)

report_wide.show(20, truncate=False)

Pivot таблица:
+----+----------+----------+----------+------+-----------+-----------+-----------+-----------+-----------+------+
|year|1         |2         |3         |4     |5          |6          |7          |8          |9          |10    |
+----+----------+----------+----------+------+-----------+-----------+-----------+-----------+-----------+------+
|2010|java      |php       |javascript|python|objective-c|c          |ruby       |delphi     |applescript|bash  |
|2011|php       |java      |javascript|python|objective-c|c          |ruby       |perl       |delphi     |bash  |
|2012|php       |javascript|java      |python|objective-c|c          |ruby       |bash       |r          |lua   |
|2013|javascript|php       |java      |python|objective-c|c          |ruby       |r          |bash       |scala |
|2014|javascript|java      |php       |python|c          |objective-c|r          |ruby       |bash       |matlab|
|2015|javascript|java      |php       |python|r          |c          |obj

In [28]:
(
    ranked_languages
    .orderBy("report_year", "place")
    .write
    .mode("overwrite")
    .parquet(PARQUET_OUT)
)

print(f"Parquet сохранён по пути: {PARQUET_OUT}")

Parquet сохранён по пути: /content/top_languages_2010_2020.parquet


In [29]:
parquet_check = spark.read.parquet(PARQUET_OUT)
parquet_check.orderBy("report_year", "place").show(50, truncate=False)

+-----------+-----------+--------+-----+
|report_year|language   |mentions|place|
+-----------+-----------+--------+-----+
|2010       |java       |52      |1    |
|2010       |php        |46      |2    |
|2010       |javascript |44      |3    |
|2010       |python     |26      |4    |
|2010       |objective-c|23      |5    |
|2010       |c          |20      |6    |
|2010       |ruby       |12      |7    |
|2010       |delphi     |8       |8    |
|2010       |applescript|3       |9    |
|2010       |bash       |3       |10   |
|2011       |php        |102     |1    |
|2011       |java       |93      |2    |
|2011       |javascript |83      |3    |
|2011       |python     |37      |4    |
|2011       |objective-c|34      |5    |
|2011       |c          |24      |6    |
|2011       |ruby       |20      |7    |
|2011       |perl       |9       |8    |
|2011       |delphi     |8       |9    |
|2011       |bash       |7       |10   |
|2012       |php        |154     |1    |
|2012       |jav

In [30]:
shutil.make_archive(
    "top_languages_2010_2020",
    'zip',
    "/content/top_languages_2010_2020.parquet"
)

files.download("top_languages_2010_2020.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
spark.stop()